In [1]:
import anndata as ad
import numpy as np
import pandas as pd

In [2]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import ConvertToNumpyArray

In [3]:
import pickle

In [4]:
import pubchempy as pcp
from tqdm import tqdm

In [6]:
def smiles_to_fingerprints(smiles_list, radius=1, fp_size=2000):
    """Convert a list of SMILES strings to Morgan (ECFP) fingerprint matrix.

    Args:
        smiles_list: Iterable of SMILES strings.
        radius: Morgan fingerprint radius (default 1, i.e. ECFP2).
        fp_size: Fingerprint bit vector size.

    Returns:
        np.ndarray of shape (len(smiles_list), fp_size), dtype uint8.
        Invalid SMILES yield a zero vector for that row.
    """
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_size)
    fps = []
    for smile in smiles_list:
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            fps.append(None)
        else:
            fp = gen.GetFingerprint(mol)
            arr = np.zeros(fp_size, dtype=np.uint8)
            ConvertToNumpyArray(fp, arr)
            fps.append(arr)
    return fps

In [7]:
tahoe_sci_op3_updated = pd.read_pickle('../../tahoe_sci_op3_updated.pkl')

In [8]:
sci = tahoe_sci_op3_updated[tahoe_sci_op3_updated['dataset'] == 'sciplex3'].reset_index(drop=True)

In [9]:
sci = sci.drop(columns=['cmap_name', 'symbol', 'code', 'symbol_', 'LPM_emb'])

In [10]:
sci['pubchem_cid'] = sci['pubchem_cid'].astype(str)

In [11]:
import os
os.makedirs('./sci_single_run_all_l1000_25_50_epochs/', exist_ok=True)
epoch_dirs = os.listdir('../../../lpm_style/files/single_run_all_50_epochs/')
epoch_dirs.remove('last')

for epoch_dir in epoch_dirs:
    df_pert_all = pd.read_pickle(f'../../../lpm_style/files/single_run_all_50_epochs/{epoch_dir}/df_pert.pkl')\
                    .rename(columns={'symbol': 'symbol_all', 
                                     'code': 'code_all', 
                                     'lpm_style_embeddings': 
                                     'lpm_style_embeddings_all'})
    df_pert_l1000 = pd.read_pickle(f'../../../lpm_style/files/single_run_l1000_50_epochs/{epoch_dir}/df_pert.pkl')\
                    .rename(columns={'symbol': 'symbol_l1000', 
                                     'code': 'code_l1000', 
                                     'lpm_style_embeddings': 
                                     'lpm_style_embeddings_l1000'})
    
    sci_merged = sci.merge(df_pert_all, left_on='pubchem_cid', right_on='symbol_all', how='left')\
                    .merge(df_pert_l1000, left_on='pubchem_cid', right_on='symbol_l1000', how='left')
    
    sci_merged['pubchem_cid'] = sci_merged['pubchem_cid'].astype(int)
    tag = int(epoch_dir.split('_')[-1])
    sci_merged.to_pickle(f"./sci_single_run_all_l1000_25_50_epochs/sci_lpm_style_embeddings_epoch_{tag + 1}.pkl")